# 🎙️ Server VoxCPM untuk PippitLokal (GPU Gratis)

Jalankan **Runtime → Change runtime type → T4 GPU**, lalu **Runtime → Run all**.
Tunggu sampai muncul **URL SERVER** (…trycloudflare.com/tts), salin ke PippitLokal.


### 1) Install

In [4]:
!pip -q install voxcpm soundfile


### 2) Tulis file server VoxCPM
(otomatis, tidak perlu upload apa pun)

In [5]:
%%writefile voxcpm_server.py
# -*- coding: utf-8 -*-
"""
voxcpm_server.py — server suara VoxCPM untuk PippitLokal.

Endpoint:
  GET  /health  -> "ok" (hanya 200 SETELAH model termuat)
  POST /tts     -> JSON {text, ref_audio_b64, speaker_wav_b64} ; balas audio/wav
"""
import os, io, sys, json, base64, tempfile, traceback, subprocess
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
import soundfile as sf
from voxcpm import VoxCPM

MODEL_ID = os.environ.get("VOXCPM_MODEL", "openbmb/VoxCPM2")
DEVICE   = os.environ.get("VOXCPM_DEVICE", "auto")
PORT     = int(os.environ.get("VOXCPM_PORT", "8081"))

print(f"[VoxCPM] Memuat model {MODEL_ID} (device={DEVICE}) ...", flush=True)
try:
    MODEL = VoxCPM.from_pretrained(MODEL_ID, load_denoiser=False, device=DEVICE)
    try:
        MODEL.generate(text="Warmup test audio")
    except Exception:
        pass
    print("[VoxCPM] Model siap 100% untuk voice cloning!", flush=True)
except Exception:
    print("[VoxCPM] GAGAL memuat model:", flush=True)
    traceback.print_exc()
    sys.exit(1)

def _synth(text, prompt_text, ref_wav_path):
    if ref_wav_path and os.path.exists(ref_wav_path) and os.path.getsize(ref_wav_path) > 100:
        print(f"[VoxCPM] Memproses AI Voice Cloning dengan sampel suara ({os.path.getsize(ref_wav_path)} bytes)...", flush=True)
        try:
            # 1. VoxCPM2 similarity voice cloning call
            wav = MODEL.generate(text=text, reference_wav_path=ref_wav_path)
            print("[VoxCPM] SUKSES: Suara dubbing berhasil dikloning persis sampel suara Anda!", flush=True)
            return _to_wav(wav)
        except Exception as e1:
            print(f"[VoxCPM] reference_wav_path gagal ({e1}), mencoba prompt_wav_path...", flush=True)
            try:
                wav = MODEL.generate(text=text, prompt_wav_path=ref_wav_path)
                print("[VoxCPM] SUKSES: Voice cloning berhasil via prompt_wav_path!", flush=True)
                return _to_wav(wav)
            except Exception as e2:
                print(f"[VoxCPM] ERROR CRITICAL VoxCPM: {e2}", flush=True)
                raise e2
    else:
        print("[VoxCPM] PERINGATAN: Sampel suara tidak ditemukan!", flush=True)
        raise ValueError("File sampel suara kosong atau tidak ditemukan oleh server VoxCPM.")

def _to_wav(wav):
    buf = io.BytesIO()
    sf.write(buf, wav, MODEL.tts_model.sample_rate, format="WAV")
    return buf.getvalue()

class H(BaseHTTPRequestHandler):
    def log_message(self, *a): pass

    def do_GET(self):
        if self.path.rstrip("/") in ["/health", "", "/tts", "/clone", "/lipsync", "/avatar"]:
            self.send_response(200); self.end_headers(); self.wfile.write(b"ok")
        else:
            self.send_response(404); self.end_headers()

    def do_POST(self):
        req_path = self.path.rstrip("/")

        if req_path not in ["/tts", "/clone", "/lipsync", "/avatar", ""]:
            self.send_response(404)
            self.end_headers()
            return

        if req_path in ["/lipsync", "/avatar"]:
            self.send_response(200)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(json.dumps({"status": "ok", "message": "Avatar photo overlay handled locally by desktop client"}).encode())
            return

        try:
            n = int(self.headers.get("Content-Length", 0))
            data = json.loads(self.rfile.read(n) or b"{}")
            text = data.get("text", "")
            prompt_text = data.get("prompt_text", "")
            ref_b64 = data.get("ref_audio_b64") or data.get("speaker_wav_b64")
            ref_path = None
            if ref_b64:
                raw_tmp = tempfile.NamedTemporaryFile(suffix=".raw", delete=False)
                raw_tmp.write(base64.b64decode(ref_b64))
                raw_tmp.close()

                clean_wav = tempfile.NamedTemporaryFile(suffix=".wav", delete=False).name
                # Convert ANY audio format (MP3, M4A, OGG, WAV) to 100% clean 16kHz Mono PCM WAV via FFmpeg
                subprocess.run(["ffmpeg", "-y", "-i", raw_tmp.name, "-ar", "16000", "-ac", "1", clean_wav], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

                try: os.unlink(raw_tmp.name)
                except: pass

                if os.path.exists(clean_wav) and os.path.getsize(clean_wav) > 100:
                    ref_path = clean_wav

            audio = _synth(text, prompt_text, ref_path)
            if ref_path and os.path.exists(ref_path):
                try: os.unlink(ref_path)
                except: pass
            self.send_response(200)
            self.send_header("Content-Type", "audio/wav")
            self.send_header("Content-Length", str(len(audio)))
            self.end_headers()
            self.wfile.write(audio)
        except Exception as e:
            traceback.print_exc()
            msg = json.dumps({"error": str(e)}).encode()
            self.send_response(500)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(msg)

if __name__ == "__main__":
    print(f"[VoxCPM] server jalan di http://0.0.0.0:{PORT}  (POST /tts, GET /health)", flush=True)
    ThreadingHTTPServer(("0.0.0.0", PORT), H).serve_forever()

Overwriting voxcpm_server.py


### 3) Jalankan server + buka URL publik gratis (cloudflared)
Biarkan sel ini terus berjalan selama memakai PippitLokal. Memuat model pertama kali butuh ~1-2 menit.

In [ ]:
import os, subprocess, time, re, urllib.request, http.client

# ====== PILIH MODEL ======
# Free T4 sering kehabisan VRAM dgn VoxCPM2. Kalau server crash karena OOM,
# ganti baris di bawah ke "openbmb/VoxCPM-0.5B" (ringan, ~5GB VRAM, cepat termuat).
os.environ["VOXCPM_MODEL"]  = os.environ.get("VOXCPM_MODEL", "openbmb/VoxCPM2")
os.environ["VOXCPM_DEVICE"] = os.environ.get("VOXCPM_DEVICE", "auto")
os.environ["VOXCPM_PORT"]   = "8081"

# unduh cloudflared (tunnel gratis, tanpa daftar)
if not os.path.exists("cloudflared"):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "cloudflared")
    os.chmod("cloudflared", 0o755)

# jalankan server VoxCPM di background, log -> server.log (juga ke layar)
logf = open("server.log", "w")
srv = subprocess.Popen(["python", "-u", "voxcpm_server.py"],
                       stdout=logf, stderr=subprocess.STDOUT)

# tunggu server sehat — sampai 12 menit (download model pertama kali bisa lama)
print("Memuat model & menunggu server sehat (maks ~12 menit) ...")
ready = False
deadline = time.time() + 12*60
last_log_size = 0
while time.time() < deadline:
    # cek apakah proses server sudah mati (crash)
    if srv.poll() is not None:
        print("\n❌ Server BERHENTI (crash). Log lengkap:\n")
        print(open("server.log").read())
        raise SystemExit("Server VoxCPM gagal start — lihat traceback di atas.")
    # tampilkan log baru
    try:
        with open("server.log") as f:
            f.seek(last_log_size); chunk = f.read(); last_log_size = f.tell()
        if chunk.strip():
            print(chunk, end="")
    except FileNotFoundError:
        pass
    # cek health
    try:
        c = http.client.HTTPConnection("127.0.0.1", 8081, timeout=3)
        c.request("GET", "/health"); r = c.getresponse()
        if r.status == 200:
            ready = True; print("\n✅ Server VoxCPM siap."); break
    except Exception:
        pass
    time.sleep(5)

if not ready:
    print("\n⚠️ Server belum siap setelah 12 menit. Log:\n")
    print(open("server.log").read())
    raise SystemExit("Timeout — coba ganti VOXCPM_MODEL ke openbmb/VoxCPM-0.5B.")

# buka tunnel ke port 8081 dan cetak URL (HANYA setelah server sehat)
tun = subprocess.Popen(["./cloudflared","tunnel","--url","http://localhost:8081"],
                       stderr=subprocess.PIPE, text=True)
url = None
for line in tun.stderr:
    print(line.strip())
    m = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        print("\n==============================================")
        print(">>> URL SERVER (Voice Clone & Reaction Avatar):")
        print(">>>", url)
        print("==============================================")
        break

print("\nBiarkan sel ini TERUS BERJALAN. Jangan tutup tab / biarkan idle,")
print("karena quick tunnel trycloudflare akan mati (URL jadi 404).")

Memuat model & menunggu server sehat (maks ~12 menit) ...
[VoxCPM] Memuat model openbmb/VoxCPM2 (device=auto) ...

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Fetching 9 files: 100%|██████████| 9/9 [01:01<00:00,  6.88s/it]
voxcpm_model_path: /root/.cache/huggingface/hub/models--openbmb--VoxCPM2/snapshots/bffb3df5a29440629464e5e839f4d214c8714c3d, zipenhancer_model_path: None, enable_denoiser: False
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
Loading AudioVAE from pytorch: /root/.cache/huggingface/hub/models--openbmb--VoxCPM2/snapshots/bffb3df5a29440629464e5e839f4d214c8714c3d/audiovae.pth
Running on device: cuda, dtype: bfloat16
Loading model from safetensor

### Catatan
- Salin **URL SERVER** ke kolom *URL server VoxCPM* di PippitLokal, lalu klik **Tes Koneksi Server**.
- **Kalau server crash (OOM) atau timeout**: di Sel 3 (runner) ganti model ke `openbmb/VoxCPM-0.5B` (ringan, ~5GB VRAM, cocok untuk T4 gratis).
- Quick tunnel trycloudflare bersifat sementara: kalau Colab idle/putus, URL mati (jadi 404). Jalankan ulang sel runner untuk dapat URL baru, lalu tempel ulang ke PippitLokal.
